# Kuroshio Daily R4 — 1993–2025 多年滚动后报正式训练

**适用平台：RTX 5080 / Ubuntu 24.04 WSL / VS Code Notebook。** 本实验不会从互联网上下载数据。

- 输入：人工 QC 冻结后的 `daily_dual_branch_paths_1993_2025_FINAL.npz`，并核查 `LONG_DAILY_1993_2025_HANDOFF.json` / 逐日 QC 状态。
- 任务：180 天真实路径历史 → 第 30 / 60 / 90 天路径。所有输入缺日只允许 **causal forward fill**（最多两天）；预测目标必须为实测/重分析派生的有效流轴，绝不补值。
- 时间泛化：按 2005 / 2009 / 2013 / 2017 / 2021 / 2024 年构造 **6 个 expanding-window rolling hindcasts**；验证区间为各测试年的上一年，且训练标签与验证标签都必须先于下一阶段的起报日期。不能跨时间随机划分。
- 主模型：保持原文 dual-branch + EOF-TCN-LSTM；基线：Persistence、季节气候态、递推 AR-EOF、**参数量匹配 LSTM**。
- 每个神经模型每折 5 seeds；结果按 fold、issue、lead、seed 保存。先进行强制 smoke / leakage / variance Gate，再允许正式训练。
- 结果：中文 CSV/JSON；原稿风格中英文 PNG（300 dpi）和矢量 PDF；训练权重、预处理参数和可恢复的预测文件。

**重要：** Uploaded Notebook 本身可能不包含运行输出。本 Notebook 在用户的 WSL 现场检查真正的数据文件，不会仅凭 Notebook 代码推断全量数据已冻结。

这组实验支持多年时间泛化，但仍需另做**独立流轴物理参考验证**；训练测试均为 CMEMS 重分析派生路径，不可称为独立观测真值。

In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R4_正式多年滚动后报'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = True   # 必须先通过 Cell 5 和 Cell 9 的资格Gate
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')

In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)

In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)

In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])

In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)

In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')

In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)

In [ ]:
# ===== 7. 五种基线/评估的科学定义（季节态和递推AR-EOF仅使用训练日期）=====
def persistence_pred(ix):
    return np.repeat(A1[ix,None,:],3,axis=1).astype(np.float32),np.repeat(A2[ix,None,:],3,axis=1).astype(np.float32)

def climatology_model(raw,trainmask):
    ids=np.flatnonzero(trainmask);dates_train=cal[ids]
    doy=np.array([t.dayofyear for t in dates_train],int)
    clims=[];glob=np.mean(raw[ids],axis=0)
    for day in range(1,367):
        cyc=np.minimum(np.abs(doy-day),366-np.abs(doy-day))
        ids2=ids[cyc<=cfg.CLIM_HALF_WINDOW_DAYS]
        clims.append(np.mean(raw[ids2],axis=0) if len(ids2)>=cfg.CLIM_MIN_SAMPLES else glob)
    return np.asarray(clims,np.float32)

def fit_recursive_AR(pc,rawvalid,train_cut,p,alpha):
    end=cal.searchsorted(train_cut,side='left')
    # 不把缺失日上已有causal fill的路径当成独立观测标签
    t=np.arange(p,end)
    ok=rawvalid[t].copy()
    for lag in range(1,p+1): ok &= rawvalid[t-lag]
    t=t[ok]
    if len(t)<200: raise RuntimeError('AR训练可用完整逐日连续片段不足200')
    XX=np.stack([pc[t-i] for i in range(p,0,-1)],axis=1).reshape(len(t),-1)
    yy=pc[t]
    reg=Ridge(alpha=alpha).fit(XX,yy)
    train_sd=np.std(pc[:end][rawvalid[:end]],axis=0)
    clip=5*np.maximum(train_sd,0.5)
    return reg,clip

def recursive_forecast(reg,clip,pc,issue,p):
    hist=np.asarray(pc[issue-p+1:issue+1],np.float32)
    if len(hist)!=p or not np.isfinite(hist).all(): raise RuntimeError('AR history bad')
    outs=[]
    for t in range(1,91):
        nxt=reg.predict(hist.reshape(1,-1))[0]
        nxt=np.clip(nxt,-clip,clip).astype(np.float32)
        hist=np.vstack((hist[1:],nxt))
        if t in cfg.LEADS: outs.append(nxt.copy())
    return np.stack(outs)

def tune_AR(fd,f,branch):
    pc=fd[f'p{branch}']; pp=fd[f'pp{branch}']; train_cut=f['train_cut']
    best=None
    validx=f['val'][::max(1,len(f['val'])//70)]
    for p in cfg.AR_LAGS:
        for alpha in cfg.AR_ALPHAS:
            reg,clip=fit_recursive_AR(pc,valid,train_cut,p,alpha)
            pred=np.stack([recursive_forecast(reg,clip,pc,int(i),p) for i in validx]);truth=np.stack([[pc[i+d] for d in cfg.LEADS] for i in validx])
            err=float(np.mean((pred-truth)**2))
            if best is None or err<best['val_mse']: best={'p':p,'alpha':alpha,'reg':reg,'clip':clip,'val_mse':err}
    return best

In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)

In [ ]:
# ===== 9. 单次训练资格：只做前向+反向+参数量+检查有限损失，绝不长训无资格机制 =====
seed0=cfg.SEEDS[0]
random.seed(seed0);np.random.seed(seed0);torch.manual_seed(seed0);torch.cuda.manual_seed_all(seed0)
f0=FOLDS[0];fd0=make_fold_data(f0)
qualification=[]
for name in ('Matched-LSTM','EOF-TCN-LSTM'):
    for branch in (1,2):
        net=make_net(name,branch,fd0).to(DEV)
        ds=CausalWindowDataset(f0['train'][:10],fd0[f'z{branch}'],fd0[f'p{branch}'])
        x,y=next(iter(DataLoader(ds,batch_size=4)))
        x=x.to(DEV);y=y.to(DEV);optimizer=torch.optim.AdamW(net.parameters(),lr=cfg.LR)
        optimizer.zero_grad();pred=net(x);loss=torch.nn.functional.mse_loss(pred,y)
        if not torch.isfinite(loss): raise RuntimeError(f'{name} B{branch} smoke loss nonfinite')
        loss.backward();g=sum(1 for p in net.parameters() if p.grad is not None and torch.isfinite(p.grad).all())
        if g<1: raise RuntimeError(f'{name} B{branch} gradients invalid')
        optimizer.step()
        qualification.append({'模型':name,'分支':branch,'参数量':nparams(net),'前向输出':str(tuple(pred.shape)),'初始损失':float(loss.detach().cpu()),'梯度层数':g,'GPU_GB':round(torch.cuda.memory_allocated()/1024**3,3),'状态':'PASS'})
        del net,ds,x,y,pred,loss,optimizer;torch.cuda.empty_cache()
pd.DataFrame(qualification).to_csv(DATA/'04_训练机制与GPU_资格预检.csv',index=False,encoding='utf-8-sig')
print('[RTX5080 TRAINING SMOKE GATE] ALL PASS');display(pd.DataFrame(qualification))
del fd0;gc.collect()

In [ ]:
# ===== 10. Neural训练/验证早停 + 单分支resume =====
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)

def predict_pc(net,issues,scaled,pcs):
    loader=DataLoader(CausalWindowDataset(issues,scaled,pcs),batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    chunks=[];net.eval()
    with torch.no_grad():
        for x,y in loader: chunks.append(net(x.to(DEV)).float().cpu().numpy())
    return np.concatenate(chunks,axis=0)

def train_branch(fold,fd,name,seed,branch):
    folder=fd['path']/name/f'seed_{seed}';folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt';meta=folder/f'branch{branch}_history.csv'
    seed_all(seed)
    net=make_net(name,branch,fd).to(DEV)
    if weight.exists() and meta.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        net.load_state_dict(ck['state']);net.eval()
        return net,pd.read_csv(meta),float(ck['best_val']),int(ck['epoch'])
    loader=DataLoader(CausalWindowDataset(fold['train'],fd[f'z{branch}'],fd[f'p{branch}']),batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    va=DataLoader(CausalWindowDataset(fold['val'],fd[f'z{branch}'],fd[f'p{branch}']),batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    amp=bool(cfg.USE_AMP);scaler=torch.amp.GradScaler('cuda',enabled=amp)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=5)
    best=float('inf');best_epoch=0;bad=0;history=[];start_epoch=1
    latest=folder/f'branch{branch}_latest_resume.pt'
    if latest.exists() and not cfg.FORCE_TRAIN:
        recover=torch.load(latest,map_location=DEV,weights_only=True)
        net.load_state_dict(recover['state']);opt.load_state_dict(recover['opt'])
        scheduler.load_state_dict(recover['scheduler']);scaler.load_state_dict(recover['scaler'])
        best=float(recover['best']);best_epoch=int(recover['best_epoch'])
        bad=int(recover['bad']);history=list(recover['history'])
        start_epoch=int(recover['epoch'])+1
        print('[branch resume]',name,branch,seed,'from epoch',start_epoch)
    max_epochs=cfg.EPOCHS_B1 if branch==1 else cfg.EPOCHS_B2
    for ep in range(start_epoch,max_epochs+1):
        net.train();sl=sn=0
        for x,y in loader:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x);loss=torch.nn.functional.mse_loss(pred,y)
            if not torch.isfinite(loss): raise RuntimeError(f'{name} B{branch} seed{seed} epoch{ep} nonfinite')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.)
            scaler.step(opt);scaler.update();sl+=float(loss.detach())*len(x);sn+=len(x)
        net.eval();vl=vn=0
        with torch.no_grad():
            for x,y in va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp): loss=torch.nn.functional.mse_loss(net(x),y)
                vl+=float(loss.detach())*len(x);vn+=len(x)
        v=vl/max(vn,1);scheduler.step(v)
        history.append({'epoch':ep,'train_mse':sl/max(sn,1),'val_mse':v,'lr':opt.param_groups[0]['lr']})
        if v<best-1e-7:
            best=v;best_epoch=ep;bad=0
            torch.save({'state':{k:t.detach().cpu() for k,t in net.state_dict().items()},'best_val':best,'epoch':ep,'model':name,'seed':seed,'branch':branch},weight)
        else:bad+=1
        if ep%5==0 or ep==max_epochs or bad>=cfg.PATIENCE:
            torch.save({'state':net.state_dict(),'opt':opt.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                        'best':best,'best_epoch':best_epoch,'bad':bad,'history':history,'epoch':ep},latest)
        if ep==1 or ep%10==0: print(f'{fold["year"]} {name} seed{seed} B{branch} ep{ep:03} train {history[-1]["train_mse"]:.5f} val {v:.5f}')
        if bad>=cfg.PATIENCE:break
    pd.DataFrame(history).to_csv(meta,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True);net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(history),best,best_epoch

In [ ]:
# ===== 11. 正式6个年份滚动后报；每run自动保存预测与评分，可恢复 =====
if not cfg.RUN_FULL_TRAINING: raise RuntimeError('RUN_FULL_TRAINING=False；只完成preflight，符合预期。')
MODELS=['Matched-LSTM','EOF-TCN-LSTM']
if cfg.TRAIN_OTHER_HYBRIDS: MODELS+=['EOF-Conv-LSTM','EOF-Conv-GRU','EOF-TCN-GRU']
all_metrics=[];all_runs=[]
for f in FOLDS:
    print('\n'+'#'*80,'test year',f['year'])
    fd=make_fold_data(f); ix=f['test'];trainmask=(cal<f['train_cut'])&valid
    # Baselines with all saved to same immutable file-per-run cache
    for name in ('Persistence','Climatology','AR-EOF'):
        cache=PRED/f'{f["year"]}_{name}.npz'
        if cache.exists() and not cfg.FORCE_TRAIN:
            with np.load(cache) as z:p1=z['b1'];p2=z['b2']
        elif name=='Persistence':
            p1=np.repeat(A1[ix][:,None,:],3,axis=1);p2=np.repeat(A2[ix][:,None,:],3,axis=1)
            np.savez_compressed(cache,b1=p1,b2=p2,issue_indices=ix)
        elif name=='Climatology':
            c1=climatology_model(A1,trainmask);c2=climatology_model(A2,trainmask)
            doys=np.asarray([cal[i+l].dayofyear-1 for i in ix for l in cfg.LEADS]).reshape(len(ix),3)
            p1=c1[doys];p2=c2[doys];np.savez_compressed(cache,b1=p1,b2=p2,issue_indices=ix)
        else:
            ar1=tune_AR(fd,f,1);ar2=tune_AR(fd,f,2)
            a1=np.stack([recursive_forecast(ar1['reg'],ar1['clip'],fd['p1'],int(i),ar1['p']) for i in tqdm(ix,desc='AR branch1')])
            a2=np.stack([recursive_forecast(ar2['reg'],ar2['clip'],fd['p2'],int(i),ar2['p']) for i in tqdm(ix,desc='AR branch2')])
            p1=fd['pp1'].invert(a1);p2=fd['pp2'].invert(a2)
            np.savez_compressed(cache,b1=p1,b2=p2,issue_indices=ix)
        metricfile=DATA/f'第{f["year"]}年_{name}_逐起报误差.csv'
        if not metricfile.exists() or cfg.FORCE_TRAIN: record_errors(f,name,-1,ix,p1,p2).to_csv(metricfile,index=False,encoding='utf-8-sig')
        all_metrics.append(pd.read_csv(metricfile))
    # neural training on RTX5080
    for name in MODELS:
        for seed in cfg.SEEDS:
            print('RUN:',f['year'],name,seed)
            cache=PRED/f'{f["year"]}_{name}_seed{seed}.npz';metricfile=DATA/f'第{f["year"]}年_{name}_种子{seed}_逐起报误差.csv'
            if cache.exists() and metricfile.exists() and not cfg.FORCE_TRAIN:
                with np.load(cache) as z:p1=z['b1'];p2=z['b2']
                all_metrics.append(pd.read_csv(metricfile));continue
            bmodels=[];hist=[];bestvals=[];beps=[]
            for branch in (1,2):
                m,h,v,ep=train_branch(f,fd,name,seed,branch);hist.append(h);bestvals.append(v);beps.append(ep)
                pred=predict_pc(m,ix,fd[f'z{branch}'],fd[f'p{branch}'])
                bmodels.append(fd[f'pp{branch}'].invert(pred))
                del m,pred;torch.cuda.empty_cache()
            p1,p2=bmodels
            np.savez_compressed(cache,b1=p1.astype(np.float32),b2=p2.astype(np.float32),issue_indices=ix)
            met=record_errors(f,name,seed,ix,p1,p2);met.to_csv(metricfile,index=False,encoding='utf-8-sig');all_metrics.append(met)
            all_runs.append({'测试年份':f['year'],'模型':name,'种子':seed,'branch1_best_epoch':beps[0],'branch2_best_epoch':beps[1],
                             'branch1_best_val_mse':bestvals[0],'branch2_best_val_mse':bestvals[1]})
            pd.DataFrame(all_runs).to_csv(DATA/'05_全部训练运行摘要.csv',index=False,encoding='utf-8-sig')
    del fd;gc.collect();torch.cuda.empty_cache()
    pd.concat(all_metrics,ignore_index=True).to_csv(DATA/'06_全部逐样本预测误差_中途快照.csv',index=False,encoding='utf-8-sig')
all_metrics=pd.concat(all_metrics,ignore_index=True)
all_metrics.to_csv(DATA/'06_全部逐样本预测误差.csv',index=False,encoding='utf-8-sig')
print('all rows=',len(all_metrics))

In [ ]:
# ===== 12. 论文统计：跨年汇总时RMSE=sqrt(mean(issue MSE))，不能先取根后错误平均 =====
metrics_path=DATA/'06_全部逐样本预测误差.csv'
if 'all_metrics' not in globals(): all_metrics=pd.read_csv(metrics_path)
runmetrics=all_metrics.groupby(['测试年份','模型','种子','提前期_天'],as_index=False).agg(
    RMSE_km=('横向路径误差MSE_km2',lambda x:float(np.sqrt(np.mean(x)))),
    MAE_km=('横向路径平均误差_km','mean'),P90_km=('横向路径P90_km','mean'),N_issues=('横向路径误差MSE_km2','size'))
runmetrics.to_csv(DATA/'07_各年份模型种子提前期性能.csv',index=False,encoding='utf-8-sig')
# 不把多seed当多年的独立样本；对各年份seed-average后等权汇总
foldmeans=runmetrics.groupby(['测试年份','模型','提前期_天'],as_index=False).agg(
    RMSE_km=('RMSE_km','mean'),MAE_km=('MAE_km','mean'),P90_km=('P90_km','mean'))
overall=foldmeans.groupby(['模型','提前期_天'],as_index=False).agg(
    逐年均值RMSE_km=('RMSE_km','mean'),逐年标准差RMSE_km=('RMSE_km','std'),逐年均值MAE_km=('MAE_km','mean'),测试年份数=('RMSE_km','size'))
ref=overall[overall['模型']=='Persistence'][['提前期_天','逐年均值RMSE_km']].rename(columns={'逐年均值RMSE_km':'持续性基线RMSE_km'})
overall=overall.merge(ref,on='提前期_天');overall['相对持续性技能']=1-overall['逐年均值RMSE_km']/overall['持续性基线RMSE_km']
overall.to_csv(DATA/'08_论文用各模型多年后报汇总.csv',index=False,encoding='utf-8-sig')
foldmeans.to_csv(DATA/'08_逐年模型汇总.csv',index=False,encoding='utf-8-sig')
display(overall.sort_values(['提前期_天','逐年均值RMSE_km']))

In [ ]:
# ===== 13. 不确定性：先resample测试年份，再在年份内paired circular 30d blocks；神经种子重采样 =====
def circ_idx(n,block,rng):
    block=max(1,min(n,int(block)));starts=rng.integers(0,n,size=int(np.ceil(n/block)))
    return np.concatenate([(s+np.arange(block))%n for s in starts])[:n]

def grouped_matrix(model,lead,year):
    df=all_metrics[(all_metrics['模型']==model)&(all_metrics['提前期_天']==lead)&(all_metrics['测试年份']==year)]
    piv=df.pivot_table(index='种子',columns='起报日期',values='横向路径误差MSE_km2').sort_index(axis=1)
    return piv

def boot_models(models,lead,nboot=None):
    # all models aligned by year+issue; common time block, paired seed draws where same numerical seeds exist
    B=cfg.BOOT_N if nboot is None else nboot;rng=np.random.default_rng(20261012+lead)
    years=np.asarray(cfg.TEST_YEARS);mats={}
    for year in years:
        frames={m:grouped_matrix(m,lead,year) for m in models}
        commons=set.intersection(*(set(q.columns) for q in frames.values()))
        ix=sorted(commons)
        if len(ix)<20:raise RuntimeError(f'{year} aligned issue dates too few')
        mats[year]={m:(np.asarray(q.index),q.reindex(columns=ix).to_numpy(float)) for m,q in frames.items()}
    reps=np.zeros((B,len(models)))
    for b in range(B):
        picked=rng.choice(years,size=len(years),replace=True)
        per_model={m:[] for m in models}
        for year in picked:
            frames=mats[year];n=next(iter(frames.values()))[1].shape[1]
            tt=circ_idx(n,cfg.BOOT_BLOCK_DAYS,rng)
            neural_seeds=next((seeds for seeds,arr in frames.values() if len(seeds)>1),np.array([-1]))
            ss=rng.integers(0,len(neural_seeds),len(neural_seeds)) if len(neural_seeds)>1 else None
            for m in models:
                seeds,arr=frames[m]
                if len(seeds)==1: v=arr[0:1,tt]
                elif ss is not None and np.array_equal(seeds,neural_seeds): v=arr[ss][:,tt]
                else:v=arr[rng.integers(0,len(seeds),size=len(seeds))][:,tt]
                # 同一模型每个year先对每个seed计算RMSE，再求seed平均，最后跨year平均
                per_model[m].append(float(np.mean(np.sqrt(np.mean(v,axis=1)))))
        reps[b]=[np.mean(per_model[m]) for m in models]
    return reps

MODEL_REPORT=['Persistence','Climatology','AR-EOF','Matched-LSTM','EOF-TCN-LSTM']
if cfg.TRAIN_OTHER_HYBRIDS:MODEL_REPORT+=['EOF-Conv-LSTM','EOF-Conv-GRU','EOF-TCN-GRU']
cirows=[];pairrows=[]
for lead in cfg.LEADS:
    reps=boot_models(MODEL_REPORT,lead)
    for j,m in enumerate(MODEL_REPORT):
        cirows.append({'模型':m,'提前期_天':lead,'95CI下限_km':np.percentile(reps[:,j],2.5),'95CI上限_km':np.percentile(reps[:,j],97.5),'bootstrap次数':len(reps),'时间块_天':cfg.BOOT_BLOCK_DAYS,'年份层级':len(cfg.TEST_YEARS)})
    k=MODEL_REPORT.index('EOF-TCN-LSTM')
    for j,m in enumerate(MODEL_REPORT):
        if j==k:continue
        d=reps[:,k]-reps[:,j]
        pairrows.append({'模型A':'EOF-TCN-LSTM','模型B':m,'提前期_天':lead,'A减B_RMSE_95CI下限_km':np.percentile(d,2.5),'A减B_RMSE_95CI上限_km':np.percentile(d,97.5),'P_A优于B':np.mean(d<0),'差异CI跨零':bool(np.percentile(d,2.5)<=0<=np.percentile(d,97.5))})
ci=pd.DataFrame(cirows);pairs=pd.DataFrame(pairrows)
ci.to_csv(DATA/'09_预测误差_跨年份分块Bootstrap_95CI.csv',index=False,encoding='utf-8-sig')
pairs.to_csv(DATA/'10_主模型与基线成对差异_95CI.csv',index=False,encoding='utf-8-sig')
display(ci);display(pairs)

In [ ]:
# ===== 14. 原稿海洋学风格：中英文双版论文图（透明? 默认白底、低饱和度蓝橙、细线）=====
COL={'Persistence':'#6E6E6E','Climatology':'#C88A68','AR-EOF':'#9F8EA4','Matched-LSTM':'#80ABC2','EOF-TCN-LSTM':'#376F95',
     'EOF-Conv-LSTM':'#7A9D65','EOF-Conv-GRU':'#B08661','EOF-TCN-GRU':'#7C6AA0'}
ZH={'Persistence':'持续性预测','Climatology':'季节气候态','AR-EOF':'自回归EOF','Matched-LSTM':'同参数量LSTM','EOF-TCN-LSTM':'EOF-TCN-LSTM',
    'EOF-Conv-LSTM':'EOF-Conv-LSTM','EOF-Conv-GRU':'EOF-Conv-GRU','EOF-TCN-GRU':'EOF-TCN-GRU'}

def style(language):
    plt.rcdefaults();matplotlib.rcParams['font.family']=[CJK_NAME,'DejaVu Sans'] if language=='zh' else ['DejaVu Serif']
    matplotlib.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.edgecolor':'#222222','axes.linewidth':0.8,
         'axes.grid':True,'grid.color':'#DDDDDD','grid.linestyle':':','grid.linewidth':0.55,'xtick.direction':'in','ytick.direction':'in',
         'xtick.top':True,'ytick.right':True,'pdf.fonttype':42,'axes.unicode_minus':False,'legend.frameon':False,'font.size':10})

def save_bi(fn,stemzh,stemen):
    for lang,output,stem in [('zh',FZH,stemzh),('en',FEN,stemen)]:
        style(lang);fig=fn(lang)
        for ext in ['png','pdf']:
            fig.savefig(output/f'{stem}.{ext}',dpi=cfg.FIG_DPI if ext=='png' else None,bbox_inches='tight')
        plt.close(fig)

def fig_coverage(lang):
    fig,ax=plt.subplots(figsize=(11,4.5));ax.bar(cover['年份'],cover['有效比例']*100,color='#80ABC2',width=.85)
    ax.axhline(85,color='#C88A68',ls='--',lw=1)
    ax.set_ylim(0,105);ax.set_xlabel('年份' if lang=='zh' else 'Year');ax.set_ylabel('有效路径覆盖率（%）' if lang=='zh' else 'Valid-axis coverage (%)');ax.set_title('1993–2025年日尺度流轴有效覆盖' if lang=='zh' else 'Daily Kuroshio axis availability, 1993–2025');fig.tight_layout();return fig
save_bi(fig_coverage,'图1_日尺度流轴年度覆盖率_中文','Figure1_Annual_Axis_Coverage_English')

def fig_split(lang):
    fig,ax=plt.subplots(figsize=(10,5));yy=np.arange(len(FOLDS))[::-1]
    for y,f in zip(yy,FOLDS):
        ax.plot([pd.Timestamp(1993,1,1),cal[f['train']+L].max()],[y,y],lw=8,solid_capstyle='butt',color='#376F95',label='训练标签' if lang=='zh' else 'Train labels')
        ax.plot([cal[f['val']].min(),cal[f['val']].max()],[y,y],lw=8,solid_capstyle='butt',color='#C88A68',label='验证起报' if lang=='zh' else 'Validation issues')
        ax.plot([cal[f['test']].min(),cal[f['test']].max()],[y,y],lw=8,solid_capstyle='butt',color='#7A9D65',label='测试起报' if lang=='zh' else 'Test issues')
    ax.set_yticks(yy,[str(f['year']) for f in FOLDS]);ax.set_xlabel('日期' if lang=='zh' else 'Date');ax.set_ylabel('测试年份' if lang=='zh' else 'Test year')
    handles,labels=ax.get_legend_handles_labels();dedup=dict(zip(labels,handles));ax.legend(dedup.values(),dedup.keys(),loc='upper left',ncol=3);fig.autofmt_xdate();fig.tight_layout();return fig
save_bi(fig_split,'图2_多年滚动后报严格时序划分_中文','Figure2_MultiYear_Rolling_Hindcasts_English')

def fig_baselines(lang):
    fig,ax=plt.subplots(figsize=(9,5.5))
    for m in MODEL_REPORT:
        q=overall[overall['模型']==m].sort_values('提前期_天')
        ax.plot(q['提前期_天'],q['逐年均值RMSE_km'],marker='o',markerfacecolor='white',color=COL[m],label=ZH[m] if lang=='zh' else m)
    ax.set_xticks(cfg.LEADS);ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)');ax.set_ylabel('跨年平均流轴RMSE（km）' if lang=='zh' else 'Across-year mean axis RMSE (km)')
    ax.legend(ncol=2);fig.tight_layout();return fig
save_bi(fig_baselines,'图3_主模型与四类基线对比_中文','Figure3_Proposed_Model_vs_Four_Baselines_English')

def fig_years(lang):
    fig,ax=plt.subplots(figsize=(10,5.5))
    q=foldmeans[foldmeans['提前期_天']==90]
    for m in MODEL_REPORT:
        mm=q[q['模型']==m].sort_values('测试年份');ax.plot(mm['测试年份'],mm['RMSE_km'],marker='o',color=COL[m],label=ZH[m] if lang=='zh' else m)
    ax.set_xticks(cfg.TEST_YEARS);ax.set_xlabel('未见测试年份' if lang=='zh' else 'Unseen hindcast year');ax.set_ylabel('90天流轴RMSE（km）' if lang=='zh' else '90-day axis RMSE (km)');ax.legend(ncol=2);fig.tight_layout();return fig
save_bi(fig_years,'图4_逐年90天泛化误差_中文','Figure4_Yearwise_90Day_Generalization_English')

def fig_unc(lang):
    fig,ax=plt.subplots(figsize=(9,5.5));shift=np.linspace(-1.0,1.0,len(MODEL_REPORT))
    for off,m in zip(shift,MODEL_REPORT):
        q=overall[overall['模型']==m].sort_values('提前期_天');b=ci[ci['模型']==m].sort_values('提前期_天')
        if len(q)!=3 or len(b)!=3: continue
        y=q['逐年均值RMSE_km'].to_numpy(float);lo=b['95CI下限_km'].to_numpy(float);hi=b['95CI上限_km'].to_numpy(float)
        # 匹配论文估计量：逐seed/年份RMSE平均；不得将估计量偷偷裁剪到CI内
        yy=y
        if np.any((yy<lo)|(yy>hi)):
            warnings.warn(f'{m} 当前点估计落在bootstrap CI外；请检查样本数量及估计量对齐')
        ax.errorbar(q['提前期_天'].to_numpy()+off,yy,yerr=np.vstack((np.maximum(0,yy-lo),np.maximum(0,hi-yy))),fmt='o-',capsize=2,color=COL[m],label=ZH[m] if lang=='zh' else m)
    ax.set_xticks(cfg.LEADS);ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)');ax.set_ylabel('误差与95%置信区间（km）' if lang=='zh' else 'RMSE and 95% CI (km)');ax.legend(ncol=2);fig.tight_layout();return fig
save_bi(fig_unc,'图5_预测误差跨年置信区间_中文','Figure5_Forecast_Error_Confidence_Intervals_English')
print('中文:',FZH,'英文:',FEN)

In [ ]:
# ===== 15. 保存复现清单 / 审稿人实验交接 =====
sha=hashlib.sha256();
with open(DS,'rb') as fin:
    while True:
        buf=fin.read(8*1024*1024)
        if not buf:break
        sha.update(buf)
record={'data_path':str(DS),'data_sha256':sha.hexdigest(),'data_days':len(dates),'date_min':str(dates.min().date()),'date_max':str(dates.max().date()),
        'fold_test_years':list(cfg.TEST_YEARS),'leads_days':list(cfg.LEADS),'history_days':cfg.HISTORY,
        'temporal_policy':'strict causal issue-date rolling hindcast; train target < validation origin - purge; validation target < test origin - purge',
        'train_only_scaler_eof':True,'eof_target':cfg.EOF_VARIANCE_TARGET,'backbone':'EOF-TCN-LSTM two independent branches',
        'reviewer_baselines':['Persistence','Climatology','recursive AR-EOF','parameter-matched LSTM'],'neural_seeds':list(cfg.SEEDS),
        'missing_policy':'input forward-fill <= 2 days; only actual valid target dates; no target interpolation',
        'skill_metric':'symmetric bidirectional closest-polyline mean-square cross-track approximation in km; annual RMSE separately',
        'independent_validation_status':'NOT part of this prediction experiment; separate independent observation-derived axis comparison required'}
(REP/'R4_正式多年滚动后报实验交接.json').write_text(json.dumps(record,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(record,indent=2,ensure_ascii=False))

### 本轮结束后的论文定位

完成上述测试仅能声称“在不同时期的再分析派生路径上进行多年份独立测试”，不能称为独立观测验证；真正的 independent physical-axis validation 仍应使用卫星测高/独立产品或专家复核的独立标签。

审稿时需要同时报告：数据覆盖及缺日规则、各测试年的验证/训练分界、每折训练专属 EOF 模态及解释方差、4 类基线、5-seed稳定性、逐年结果和95%CI。若模型在部分年份或提前期未优于季节气候态/AR-EOF，应如实报告，不能省略基线或只呈现有利年份。